# 🔥 Colab B — PyTorch From-Scratch 3-Layer Deep Neural Network
## Nonlinear Regression WITHOUT using PyTorch Built-in Layer Functionality

### Key Design Decisions
| Item | Choice |
|---|---|
| Framework | **PyTorch raw tensors** — no `nn.Module`, no `nn.Linear` |
| Architecture | Input(3) → Hidden1(16) → Hidden2(8) → Output(1) |
| Activation | **Swish** (manual implementation) |
| Gradient | PyTorch **autograd** on raw `torch.Tensor` with `requires_grad=True` |
| Data | Same 3-variable nonlinear: `y = sin(x1)*cos(x2) + x3²` |


In [ ]:
# ── SECTION 1: Imports ──────────────────────────────────────────────────────
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ── SECTION 2: Synthetic Data ────────────────────────────────────────────────
N = 1000
X1 = np.random.uniform(-np.pi, np.pi, N)
X2 = np.random.uniform(-np.pi, np.pi, N)
X3 = np.random.uniform(-2, 2, N)
Y  = np.sin(X1) * np.cos(X2) + X3**2 + np.random.randn(N) * 0.1

X = np.column_stack([X1, X2, X3])
scaler_X = StandardScaler()
scaler_Y = StandardScaler()
X_norm = scaler_X.fit_transform(X)
Y_norm = scaler_Y.fit_transform(Y.reshape(-1, 1))

# Convert to PyTorch tensors
X_t = torch.tensor(X_norm, dtype=torch.float32).to(device)
Y_t = torch.tensor(Y_norm, dtype=torch.float32).to(device)
print(f'X: {X_t.shape}, Y: {Y_t.shape}')

In [ ]:
# ── SECTION 3: Raw Weight Initialization (NO nn.Module) ─────────────────────
# He initialization — returns tensors with requires_grad=True

def make_weight(rows, cols):
    w = torch.randn(rows, cols, device=device) * (2.0 / rows) ** 0.5
    return w.requires_grad_(True)

def make_bias(cols):
    b = torch.zeros(1, cols, device=device)
    return b.requires_grad_(True)

# 3 → 16 → 8 → 1
W1, b1 = make_weight(3, 16),  make_bias(16)
W2, b2 = make_weight(16, 8),  make_bias(8)
W3, b3 = make_weight(8, 1),   make_bias(1)

parameters = [W1, b1, W2, b2, W3, b3]
print('Weights created:')
for p, name in zip(parameters, ['W1','b1','W2','b2','W3','b3']):
    print(f'  {name}: {tuple(p.shape)}')

In [ ]:
# ── SECTION 4: Swish Activation (Manual) ────────────────────────────────────
def swish(z):
    """Swish = z * sigmoid(z) — smooth, non-monotonic activation."""
    return z * torch.sigmoid(z)

# ── SECTION 5: Forward Pass (Pure Matrix Ops) ───────────────────────────────
def forward(X):
    """
    3-layer forward pass using raw torch matmul.
    No nn.Linear — purely manual weight application.
    """
    # Layer 1
    Z1 = X @ W1 + b1          # [N, 16]
    A1 = swish(Z1)

    # Layer 2
    Z2 = A1 @ W2 + b2         # [N, 8]
    A2 = swish(Z2)

    # Layer 3 — linear output
    Z3 = A2 @ W3 + b3         # [N, 1]
    return Z3

y_test = forward(X_t[:5])
print('Forward test:', y_test.detach().cpu().numpy().ravel())

In [ ]:
# ── SECTION 6: Training Loop with Manual Gradient Zero + Step ───────────────
EPOCHS = 2000
LR     = 0.01
loss_history = []

for epoch in range(EPOCHS):
    # --- Forward ---
    y_pred = forward(X_t)

    # --- MSE Loss ---
    loss = torch.mean((y_pred - Y_t) ** 2)
    loss_history.append(loss.item())

    # --- Backward (autograd computes gradients) ---
    loss.backward()

    # --- Manual SGD update (no optimizer object) ---
    with torch.no_grad():
        for p in parameters:
            p -= LR * p.grad
            p.grad.zero_()         # clear gradients manually

    if epoch % 200 == 0:
        print(f'Epoch {epoch:4d} | MSE Loss: {loss.item():.6f}')

print(f'\nFinal MSE: {loss_history[-1]:.6f}')

In [ ]:
# ── SECTION 7: Results ───────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(loss_history, color='darkorange')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss'); plt.yscale('log')
plt.title('Training Loss — PyTorch Scratch 3-Layer Net (No nn.Module)')
plt.grid(True); plt.show()

with torch.no_grad():
    y_pred_np = forward(X_t).cpu().numpy()
y_pred_inv = scaler_Y.inverse_transform(y_pred_np)
y_true_inv = scaler_Y.inverse_transform(Y_norm)

plt.figure(figsize=(8, 5))
plt.scatter(y_true_inv[:200], y_pred_inv[:200], alpha=0.5, s=20)
plt.plot([y_true_inv.min(), y_true_inv.max()],
         [y_true_inv.min(), y_true_inv.max()], 'r--', label='Perfect')
plt.xlabel('True y'); plt.ylabel('Predicted y')
plt.title('True vs Predicted — PyTorch Scratch 3-Layer')
plt.legend(); plt.grid(True); plt.show()

print(f'Final MSE (original scale): {np.mean((y_true_inv - y_pred_inv)**2):.4f}')